# FZ Parametric Studies with `fzr`

In this notebook we run **parametric studies** on a simply-supported beam
deflection model:

> **δ = F L³ / (48 E I)**

Topics covered:
- Grid inputs (dict with lists) → automatic Cartesian product
- DataFrame inputs (explicit scenarios)
- Progress callbacks
- Parallel calculators (race-to-finish)
- Results analysis and visualisation


In [1]:
import json, time
from pathlib import Path
import fz
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm

fz.set_log_level("WARNING")
WORK = Path("work_03_parametric")
WORK.mkdir(exist_ok=True)
(WORK / "models").mkdir(exist_ok=True)


## 1 · Beam simulation script

In [2]:
SIM = WORK / "models" / "beam.py"
SIM.write_text('''#!/usr/bin/env python3
"""Simply-supported beam: delta = F*L^3 / (48*E*I)"""
params = {}
with open("beam.in") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "=" in line:
            k, v = line.split("=", 1)
            params[k.strip()] = float(v.strip())

F = params["F"]
L = params["L"]
E = params["E"]
I = params["I"]

delta    = F * L**3 / (48 * E * I)
delta_mm = delta * 1000

with open("output.txt", "w") as fh:
    fh.write(f"deflection_m  = {delta:.8f}\\n")
    fh.write(f"deflection_mm = {delta_mm:.6f}\\n")
    fh.write("status = OK\\n")
''')
print("Simulator:", SIM)


Simulator: work_03_parametric/models/beam.py


## 2 · Template and model

In [3]:
TMPL = WORK / "beam.in"
TMPL.write_text(
    "F = ${F~10000}      # applied load (N)\n"
    "L = ${L~5.0}        # beam span (m)\n"
    "E = ${E~200e9}      # Young's modulus (Pa)\n"
    "I = ${I~8.333e-6}   # second moment of area (m^4)\n"
)

MODEL = {
    "varprefix": "$", "formulaprefix": "@",
    "delim": "{}", "commentline": "#",
    "output": {
        "deflection_m":  "grep '^deflection_m '  output.txt | awk '{print $3}'",
        "deflection_mm": "grep '^deflection_mm ' output.txt | awk '{print $3}'",
    },
}
CALC = f"sh://python3 {SIM.resolve()}"
print("Template and model ready.")


Template and model ready.


## 3 · Grid inputs — Cartesian product

In [4]:
df_grid = fz.fzr(
    str(TMPL),
    {
        "F": [5_000, 10_000, 20_000, 50_000],
        "L": [3.0, 5.0, 7.0],
        "E": [200e9],
        "I": [8.333e-6],
    },
    MODEL,
    results_dir=str(WORK / "results_grid"),
    calculators=[CALC],
)
df_grid["F"]  = df_grid["F"].astype(float)
df_grid["L"]  = df_grid["L"].astype(float)
df_grid["deflection_mm"] = pd.to_numeric(df_grid["deflection_mm"], errors="coerce")

print(f"Grid: {len(df_grid)} cases")
print(df_grid[["F", "L", "deflection_mm"]].to_string(index=False))


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/12) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/12) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/12) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/12) ETA: ...

◢ [███>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   8% (1/12) ETA: 6s

...

◥ [████████████████████████████████████>░░░]  91% (11/12) ETA: 0s

◢ [████████████████████████████████████>░░░]  91% (11/12) ETA: 0s

◣ [████████████████████████████████████>░░░]  91% (11/12) ETA: 0s

  [████████████████████████████████████████] 100% (12/12) Total: 6s

Grid: 12 cases
      F   L  deflection_mm
 5000.0 3.0       1.687568
 5000.0 5.0       7.812813
 5000.0 7.0      21.438358
10000.0 3.0       3.375135
10000.0 5.0      15.625625
10000.0 7.0      42.876715
20000.0 3.0       6.750270
20000.0 5.0      31.251250
20000.0 7.0      85.753430
50000.0 3.0      16.875675
50000.0 5.0      78.128125
50000.0 7.0     214.383575


## 4 · DataFrame inputs — explicit scenarios

Pass a `pandas.DataFrame` to specify *exact* cases (no Cartesian product).

In [5]:
scenarios = pd.DataFrame([
    {"F": 2_000,  "L": 4.0, "E": 11e9,  "I": 1.667e-6, "label": "timber_light"},
    {"F": 5_000,  "L": 6.0, "E": 11e9,  "I": 5.208e-6, "label": "timber_medium"},
    {"F": 10_000, "L": 8.0, "E": 11e9,  "I": 1.25e-5,  "label": "timber_heavy"},
    {"F": 10_000, "L": 5.0, "E": 200e9, "I": 8.333e-6, "label": "steel_HEB200"},
    {"F": 50_000, "L": 10.0,"E": 200e9, "I": 4.57e-4,  "label": "steel_HEB400"},
])
print("Scenarios:")
print(scenarios[["label", "F", "L"]].to_string(index=False))

df_scenarios = fz.fzr(
    str(TMPL),
    scenarios[["F", "L", "E", "I"]],
    MODEL,
    results_dir=str(WORK / "results_scenarios"),
    calculators=[CALC],
)
df_scenarios["deflection_mm"] = pd.to_numeric(df_scenarios["deflection_mm"], errors="coerce")
df_scenarios["label"] = scenarios["label"].values
print("\nResults:")
print(df_scenarios[["label", "F", "L", "deflection_mm"]].to_string(index=False))


Scenarios:
        label     F    L
 timber_light  2000  4.0
timber_medium  5000  6.0
 timber_heavy 10000  8.0
 steel_HEB200 10000  5.0
 steel_HEB400 50000 10.0
  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/5) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/5) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/5) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/5) ETA: ...

◢ [████████>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  20% (1/5) ETA: 2s

...

◥ [████████████████████████████████>░░░░░░░]  80% (4/5) ETA: 0s

◢ [████████████████████████████████>░░░░░░░]  80% (4/5) ETA: 0s

◣ [████████████████████████████████>░░░░░░░]  80% (4/5) ETA: 0s

  [████████████████████████████████████████] 100% (5/5) Total: 2s


Results:
        label       F    L  deflection_mm
 timber_light  2000.0  4.0     145.425460
timber_medium  5000.0  6.0     392.752409
 timber_heavy 10000.0  8.0     775.757576
 steel_HEB200 10000.0  5.0      15.625625
 steel_HEB400 50000.0 10.0      11.396791


## 5 · Progress callbacks

fzr accepts a `callbacks` dict with hooks fired at different stages.  

Signatures:
- `on_start(total_cases, calculators)`
- `on_case_start(case_index, total_cases, var_combo)`
- `on_case_complete(case_index, total_cases, var_combo, status, result)`
- `on_progress(completed, total, eta_seconds)`
- `on_complete(total_cases, completed_cases, results)`

In [6]:
from datetime import datetime

log = []

def on_start(total_cases, calculators):
    log.append(f"START  total={total_cases}  calcs={len(calculators)}")

def on_case_start(case_index, total_cases, var_combo):
    log.append(f"CASE_START  {case_index+1}/{total_cases}  {var_combo}")

def on_case_complete(case_index, total_cases, var_combo, status, result):
    log.append(f"CASE_DONE   {case_index+1}/{total_cases}  status={status}")

def on_progress(completed, total, eta_seconds):
    log.append(f"PROGRESS {completed}/{total}  eta={eta_seconds:.0f}s")

def on_complete(total_cases, completed_cases, results):
    log.append(f"COMPLETE  done={completed_cases}/{total_cases}")

fz.fzr(
    str(TMPL),
    {"F": [5_000, 10_000], "L": [3.0, 5.0], "E": [200e9], "I": [8.333e-6]},
    MODEL,
    results_dir=str(WORK / "results_callbacks"),
    calculators=[CALC],
    callbacks={
        "on_start":         on_start,
        "on_case_start":    on_case_start,
        "on_case_complete": on_case_complete,
        "on_progress":      on_progress,
        "on_complete":      on_complete,
    },
)
print("Callback log:")
for entry in log:
    print(" ", entry)


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/4) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/4) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/4) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/4) ETA: ...

◢ [██████████>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  25% (1/4) ETA: 1s

...

◢ [██████████████████████████████>░░░░░░░░░]  75% (3/4) ETA: 0s

◣ [██████████████████████████████>░░░░░░░░░]  75% (3/4) ETA: 0s

◤ [██████████████████████████████>░░░░░░░░░]  75% (3/4) ETA: 0s

  [████████████████████████████████████████] 100% (4/4) Total: 2s

Callback log:
  START  total=4  calcs=1
  CASE_START  1/4  {'F': 5000, 'L': 3.0, 'E': 200000000000.0, 'I': 8.333e-06}
  CASE_DONE   1/4  status=done
  CASE_START  2/4  {'F': 5000, 'L': 5.0, 'E': 200000000000.0, 'I': 8.333e-06}
  CASE_DONE   2/4  status=done
  CASE_START  3/4  {'F': 10000, 'L': 3.0, 'E': 200000000000.0, 'I': 8.333e-06}
  CASE_DONE   3/4  status=done
  CASE_START  4/4  {'F': 10000, 'L': 5.0, 'E': 200000000000.0, 'I': 8.333e-06}
  CASE_DONE   4/4  status=done
  COMPLETE  done=4/4


## 6 · Parallel calculators (race-to-finish)

Passing *multiple* calculators runs them **simultaneously** per case.  The first to finish wins — useful on heterogeneous clusters.

In [7]:
SIM_SLOW = WORK / "models" / "beam_slow.sh"
SIM_SLOW.write_text(f'#!/bin/bash\nsleep 1\npython3 {SIM.resolve()} "$@"\n')
SIM_SLOW.chmod(0o755)

t0 = time.time()
df_par = fz.fzr(
    str(TMPL),
    {"F": [5_000, 20_000], "L": [4.0, 6.0], "E": [200e9], "I": [8.333e-6]},
    MODEL,
    results_dir=str(WORK / "results_parallel"),
    calculators=[
        f"sh://bash {SIM_SLOW.resolve()}",   # slow (1 s extra)
        CALC,                                 # fast  (immediate)
    ],
)
elapsed = time.time() - t0
print(f"Parallel run: {elapsed:.1f}s for {len(df_par)} cases")
print(df_par[["F", "L", "deflection_mm", "calculator", "status"]].to_string(index=False))


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/4) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/4) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/4) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/4) ETA: ...

◢ [██████████>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  25% (1/4) ETA: 0s

...

◢ [████████████████████>░░░░░░░░░░░░░░░░░░░]  50% (2/4) ETA: 0s

◣ [████████████████████>░░░░░░░░░░░░░░░░░░░]  50% (2/4) ETA: 0s

◤ [████████████████████>░░░░░░░░░░░░░░░░░░░]  50% (2/4) ETA: 0s

  [████████████████████████████████████████] 100% (4/4) Total: 1s

Parallel run: 1.8s for 4 cases
    F   L  deflection_mm                                                                                               calculator status
 5000 4.0        4.00016 sh://bash /home/richet/Sync/Open/Funz/github/fz/examples/work_03_parametric/models/beam_slow.sh#8fc58d1f   done
 5000 6.0       13.50054   sh://python3 /home/richet/Sync/Open/Funz/github/fz/examples/work_03_parametric/models/beam.py#7eda624b   done
20000 4.0       16.00064   sh://python3 /home/richet/Sync/Open/Funz/github/fz/examples/work_03_parametric/models/beam.py#7eda624b   done
20000 6.0       54.00216   sh://python3 /home/richet/Sync/Open/Funz/github/fz/examples/work_03_parametric/models/beam.py#7eda624b   done


## 7 · Visualisation

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pivot for the heatmap
pivot = df_grid.pivot_table(values="deflection_mm", index="F", columns="L")
im = axes[0].imshow(pivot.values, aspect="auto", cmap="YlOrRd",
                    extent=[pivot.columns.min(), pivot.columns.max(),
                            float(pivot.index.min()), float(pivot.index.max())])
axes[0].set_xlabel("Span L (m)"); axes[0].set_ylabel("Load F (N)")
axes[0].set_title("Beam deflection (mm)"); plt.colorbar(im, ax=axes[0], label="mm")

colours = cm.Set2(range(len(df_scenarios)))
axes[1].barh(df_scenarios["label"], df_scenarios["deflection_mm"], color=colours)
axes[1].set_xlabel("Deflection (mm)")
axes[1].set_title("Beam deflection by scenario"); axes[1].grid(axis="x", alpha=.3)

plt.tight_layout()
plt.savefig(str(WORK / "beam_results.png"), dpi=100)
plt.show()
print("Plot saved.")


Plot saved.


/tmp/ipykernel_51175/953118721.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
